# A549 LOY and TreeMet Analysis

Reviewer-ready analysis of LOY/ROY and TreeMet metrics using the integrated dataset.


## 1. Environment and versions

In [ ]:
# Environment + determinism (set before importing heavy libs)
import os

# Best-effort determinism across machines (threads can introduce tiny non-determinism)
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')
os.environ.setdefault('VECLIB_MAXIMUM_THREADS', '1')
os.environ.setdefault('NUMBA_NUM_THREADS', '1')

import scanpy as sc
import harmonypy
import leidenalg
import igraph
import umap

from importlib.metadata import PackageNotFoundError, version


def _pkg_version(dist_name: str, module=None) -> str:
    try:
        return version(dist_name)
    except PackageNotFoundError:
        return getattr(module, "__version__", "unknown")


sc.settings.n_jobs = 1
sc.logging.print_header()
print(f"scanpy: {_pkg_version('scanpy', sc)}")
print(f"harmonypy: {_pkg_version('harmonypy', harmonypy)}")
print(f"leidenalg: {_pkg_version('leidenalg', leidenalg)}")
print(f"python-igraph: {_pkg_version('igraph', igraph)}")
print(f"umap-learn: {_pkg_version('umap-learn', umap)}")


## 2. Parameters

In [ ]:
# Scoring
MIN_GENES_FOR_SCORING = 3
CTRL_SIZE = 50

# LOY/ROY
LOY_THRESHOLD = -0.2
GROUP_KEY = 'Sample2'
REFERENCE_GROUP = 'LL'

# Correlation plots
CORR_METHOD = 'spearman'
X_LIMITS = (-0.5, 0.7)
Y_LIMITS = (-0.1, 0.6)
EXCLUDE_GROUPS = ['LM0_Engraft', 'LM0_NotEngraft']

# Reproducibility
RANDOM_SEED = 0
import random
import numpy as np
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


## 3. Paths

In [ ]:
from pathlib import Path
import sys

CWD = Path.cwd().resolve()
for p in [CWD, *CWD.parents]:
    if (p / "code" / "figure6" / "scripts" / "project_paths.py").exists():
        REPO_ROOT = p
        break
else:
    raise RuntimeError(
        "Could not locate the repository root. Launch Jupyter from within the repo (or a subdirectory) so code/figure6/notebooks is visible."
    )

if str(REPO_ROOT / "code" / "figure6") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "code" / "figure6"))

from scripts.project_paths import get_notebook_paths

PATHS = get_notebook_paths("02_LOY_TreeMet", REPO_ROOT)
PROJECT_ROOT = PATHS.common.repo_root
DIR_CODE = PATHS.common.code_dir
DIR_DATA = PATHS.common.data_dir
DIR_RUNTIME = PATHS.common.runtime_dir
DIR_FIGURES_ROOT = PATHS.common.figures_root
DIR_SAVE = PATHS.save_dir
DIR_FIG_SAVE = PATHS.figure_dir
DIR_PROC_IN = PATHS.common.processed_dir

sc.settings.verbosity = 3
sc.settings.figdir = str(DIR_FIG_SAVE)


## 4. Imports and helper functions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scripts.utils import (
    add_loy_roy_labels,
    add_loy_roy_proportions,
    build_loy_roy_table,
    chi_square_loy_roy,
    fisher_pairwise_vs_reference,
    plot_score_correlation,
)


def my_savefig(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    pdf_path = path.with_suffix('.pdf')
    png_path = path.with_suffix('.png')
    plt.savefig(pdf_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.savefig(png_path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"[Saved] {pdf_path}")
    print(f"[Saved] {png_path}")


## 5. Load integrated dataset

In [ ]:
adata = sc.read(DIR_PROC_IN / 'A549_integrated_final.h5ad')
print(adata)


## 6. Gene signature scoring
Score Y, X, and housekeeping gene sets.

In [ ]:
GENE_SIGNATURES = {
    'Y_score': [
        'EIF1AY','KDM5D','TXLNGY','RPS4Y1','UTY','DDX3Y','ZFY','PRKY','TBL1Y',
        'NLGN4Y','USP9Y','TMSB4Y','PCDH11Y'
    ],
    'X_score': [
        'EIF1AX','KDM5C','TXLNG','RPS4X','KDM6A','DDX3X','ZFX','PRKX','TBL1X',
        'NLGN4X','USP9X','TMSB4X','PCDH11X'
    ],
    'Housekeep_score': [
        'RPLP0', 'RPL13A', 'RPS29', 'RPS27', 'RPS12',
        'EEF1A1', 'ACTB', 'GAPDH', 'PSMB2', 'HPRT1', 'UBC'
    ]
}

# Use raw if available
use_raw = adata.raw is not None
gene_universe = set(adata.raw.var_names if use_raw else adata.var_names)

scored_keys = []
for score_name, genes in GENE_SIGNATURES.items():
    present = [g for g in genes if g in gene_universe]
    if len(present) < MIN_GENES_FOR_SCORING:
        continue
    sc.tl.score_genes(
        adata,
        gene_list=present,
        ctrl_size=min(CTRL_SIZE, len(present)),
        score_name=score_name,
        use_raw=use_raw,
    )
    scored_keys.append(score_name)

print(f"Scored: {scored_keys}")


## 8. LOY/ROY classification
Label LOY/ROY and summarize proportions.

In [ ]:
add_loy_roy_labels(
    adata,
    threshold=LOY_THRESHOLD,
    label_col='Y_group',
    low_label='LOY',
    high_label='ROY',
)

counts = build_loy_roy_table(
    adata,
    group_key=GROUP_KEY,
    label_col='Y_group',
    low_label='LOY',
    high_label='ROY',
)
props = add_loy_roy_proportions(counts, low_label='LOY', high_label='ROY')

props.to_csv(DIR_SAVE / 'loy_roy_proportions_by_group.csv')
print(props.head())


## Figure (Main): Panels D-F
Generate the three main-panel figures used in the manuscript (D/E/F).

- Panels D/E remain all-cell compartment summaries.
- Panel F follows the historical lineage-enriched LOY/ROY definition traced from `05_Replot_new`.
- Each figure is exported in both `pdf` and `png` format under `figures/figure6/02_LOY_TreeMet/`.


In [ ]:
import json
import seaborn as sns

# ---- Figure settings ----
LOY_COLOR = 'tab:blue'
ROY_COLOR = 'tab:orange'
LOYROY_PALETTE = {'LOY': LOY_COLOR, 'ROY': ROY_COLOR}

# Historical Fig 6G/F lineage-enriched rule traced from 05_Replot_new/gg.ipynb
LINEAGE_MIN_CELLS = 10
LINEAGE_CLASS_CUTOFF = 0.6
LINEAGE_PRIMARY_SITE = 'LL'
LINEAGE_SITE_RULE = 'primary_only'
LINEAGE_USE_MIN_CELLS_PER_SITE = False
LINEAGE_MIN_CELLS_PER_SITE = 0

# Map abbreviations to manuscript-friendly labels
COMPARTMENT_LABELS = {
    'LL': 'Left lung',
    'RL': 'Right lung',
    'Liv': 'Liver',
    'M': 'Lymph node',
    'LM0_Engraft': 'Engrafted',
    'LM0_NotEngraft': 'Not engrafted',
}
# Ordering differs by panel to match the manuscript figure style
COMPARTMENT_ORDER_D = ['LL', 'Liv', 'M', 'RL', 'LM0_Engraft', 'LM0_NotEngraft']
# Fig E (stacked bar) puts LM0 first, then metastatic sites
COMPARTMENT_ORDER_E = ['LM0_NotEngraft', 'LM0_Engraft', 'LL', 'RL', 'M', 'Liv']

required_cols = [GROUP_KEY, 'LineageGroup', 'TreeMetRate', 'Y_score', 'Y_group']
missing_cols = [col for col in required_cols if col not in adata.obs.columns]
if missing_cols:
    raise KeyError(f'Missing required columns in adata.obs: {missing_cols}')

# Use scTreeMetRate as the metastatic capacity metric (paper label: scMetRate).
if 'scMetRate' not in adata.obs.columns:
    if 'scTreeMetRate' not in adata.obs.columns:
        raise KeyError('Expected scTreeMetRate in adata.obs')
    adata.obs['scMetRate'] = adata.obs['scTreeMetRate']

plot_df_all = adata.obs[[GROUP_KEY, 'LineageGroup', 'TreeMetRate', 'Y_score', 'scMetRate', 'Y_group']].copy()
plot_df_all[GROUP_KEY] = plot_df_all[GROUP_KEY].astype(str)
plot_df_all['Compartment'] = plot_df_all[GROUP_KEY].map(COMPARTMENT_LABELS).fillna(plot_df_all[GROUP_KEY])
plot_df_all['LineageGroup'] = plot_df_all['LineageGroup'].astype(float).fillna(-1).astype(int).astype(str)

# Keep only known compartments (stable ordering) using the all-cell dataframe.
present = set(plot_df_all[GROUP_KEY].unique())
comp_order_D = [COMPARTMENT_LABELS[k] for k in COMPARTMENT_ORDER_D if k in present]
comp_order_E = [COMPARTMENT_LABELS[k] for k in COMPARTMENT_ORDER_E if k in present]
comp_palette = {
    COMPARTMENT_LABELS[k]: c
    for k, c in zip(
        [k for k in COMPARTMENT_ORDER_D if k in present],
        sns.color_palette('tab10', n_colors=len([k for k in COMPARTMENT_ORDER_D if k in present])),
    )
}

# Rebuild the historical lineage-enriched LOY/ROY classification used for the old Figure 6G/F.
lineage_df = plot_df_all[
    ~plot_df_all[GROUP_KEY].isna()
    & ~plot_df_all['TreeMetRate'].isna()
    & ~plot_df_all['scMetRate'].isna()
    & (plot_df_all['LineageGroup'] != '0')
    & (plot_df_all['LineageGroup'] != '-1')
].copy()

lineage_counts = lineage_df['LineageGroup'].value_counts()
valid_lineages = set(lineage_counts[lineage_counts >= LINEAGE_MIN_CELLS].index)
lineage_df = lineage_df[lineage_df['LineageGroup'].isin(valid_lineages)].copy()

lr_site_full = (
    lineage_df.groupby(['LineageGroup', GROUP_KEY, 'Y_group'], observed=True)
    .size()
    .unstack(fill_value=0)
)
for col in ['LOY', 'ROY']:
    if col not in lr_site_full.columns:
        lr_site_full[col] = 0
lr_site_full['n_total'] = lr_site_full['LOY'] + lr_site_full['ROY']
lr_site_full['loy_pct'] = lr_site_full['LOY'] / lr_site_full['n_total'].replace(0, np.nan)
lr_site_full['roy_pct'] = lr_site_full['ROY'] / lr_site_full['n_total'].replace(0, np.nan)

if LINEAGE_USE_MIN_CELLS_PER_SITE:
    lr_site = lr_site_full[lr_site_full['n_total'] >= LINEAGE_MIN_CELLS_PER_SITE].copy()
else:
    lr_site = lr_site_full.copy()

site_sets = lr_site.reset_index().groupby('LineageGroup')[GROUP_KEY].apply(set)
eligible_lineages = {
    lg for lg, sites in site_sets.items()
    if (LINEAGE_PRIMARY_SITE in sites and any(site != LINEAGE_PRIMARY_SITE for site in sites))
}

def assign_lineage_class(lineage: str) -> str:
    if lineage not in eligible_lineages:
        return 'Ineligible'
    sub = lr_site.xs(lineage, level='LineageGroup')
    sub = sub[sub['n_total'] > 0].copy()
    if LINEAGE_SITE_RULE == 'primary_only':
        if LINEAGE_PRIMARY_SITE not in sub.index:
            return 'Ineligible'
        sub_check = sub.loc[[LINEAGE_PRIMARY_SITE]]
    else:
        sub_check = sub

    loy_ok = (sub_check['loy_pct'] >= LINEAGE_CLASS_CUTOFF).all()
    roy_ok = (sub_check['roy_pct'] >= LINEAGE_CLASS_CUTOFF).all()
    if loy_ok and not roy_ok:
        return 'LOY'
    if roy_ok and not loy_ok:
        return 'ROY'
    return 'Mixed'

lineage_class = {
    lineage: assign_lineage_class(lineage)
    for lineage in lr_site.index.get_level_values('LineageGroup').unique()
}
plot_df_all['LineageClass'] = plot_df_all['LineageGroup'].map(lineage_class).fillna('Ineligible')

site_summary = lr_site_full.reset_index().rename(columns={'LOY': 'loy', 'ROY': 'roy'})
site_summary['eligible'] = site_summary['LineageGroup'].isin(eligible_lineages)
site_summary['lineageClass'] = site_summary['LineageGroup'].map(lineage_class).fillna('Ineligible')
site_summary['pass_min_cells'] = site_summary['n_total'] >= LINEAGE_MIN_CELLS_PER_SITE
site_summary['used_for_enrichment'] = True if not LINEAGE_USE_MIN_CELLS_PER_SITE else site_summary['pass_min_cells']
site_summary['LineageGroup'] = site_summary['LineageGroup'].astype(int)
site_summary = site_summary[[
    'LineageGroup', GROUP_KEY, 'loy', 'roy', 'n_total', 'loy_pct', 'roy_pct',
    'lineageClass', 'eligible', 'pass_min_cells', 'used_for_enrichment',
]].sort_values(['LineageGroup', GROUP_KEY], kind='stable')
site_summary.to_csv(DIR_SAVE / 'LOYROY_site_summary.tsv', sep='\t', index=False)
site_summary[site_summary['eligible']].to_csv(DIR_SAVE / 'LOYROY_site_summary_eligible.tsv', sep='\t', index=False)

figf_df = plot_df_all[plot_df_all['LineageClass'].isin(['LOY', 'ROY'])].copy()

sns.set_theme(style='whitegrid', context='paper')

# ---- Plot-ready exports (for Prism/Origin/R, etc.) ----
DIR_PLOT_DATA = DIR_SAVE / 'plot_data'
DIR_PLOT_DATA.mkdir(parents=True, exist_ok=True)

# Single tidy table sufficient to re-plot Fig D/E/F.
# Y_group is per-cell; LineageClass carries the historical lineage-enriched class for Fig F.
plot_df_export = plot_df_all[[
    'Compartment', 'LineageGroup', 'Y_group', 'LineageClass', 'Y_score', 'scMetRate'
]].copy()
plot_df_export.to_csv(DIR_PLOT_DATA / 'Fig6DEF_per_cell.csv', index=True, index_label='cell')
figf_df[['Compartment', 'LineageGroup', 'LineageClass', 'scMetRate']].to_csv(
    DIR_PLOT_DATA / 'FigF_scMetRate_lineage_enriched_per_cell.csv',
    index=True,
    index_label='cell',
)

# Label mapping + intended x-axis orders (helps reviewers reproduce the same ordering).
(DIR_PLOT_DATA / 'Fig6DEF_metadata.json').write_text(json.dumps({
    'COMPARTMENT_LABELS': COMPARTMENT_LABELS,
    'COMPARTMENT_ORDER_D': COMPARTMENT_ORDER_D,
    'COMPARTMENT_ORDER_E': COMPARTMENT_ORDER_E,
    'comp_order_D_labels': comp_order_D,
    'comp_order_E_labels': comp_order_E,
    'LOY_THRESHOLD': float(LOY_THRESHOLD),
    'LINEAGE_MIN_CELLS': int(LINEAGE_MIN_CELLS),
    'LINEAGE_CLASS_CUTOFF': float(LINEAGE_CLASS_CUTOFF),
    'LINEAGE_PRIMARY_SITE': LINEAGE_PRIMARY_SITE,
    'LINEAGE_SITE_RULE': LINEAGE_SITE_RULE,
    'LINEAGE_USE_MIN_CELLS_PER_SITE': bool(LINEAGE_USE_MIN_CELLS_PER_SITE),
    'LINEAGE_MIN_CELLS_PER_SITE': int(LINEAGE_MIN_CELLS_PER_SITE),
    'FigF_definition': 'historical lineage-enriched LOY/ROY class from 05_Replot_new',
}, indent=2, sort_keys=True))

# (D) Y-score violin by compartment
fig, ax = plt.subplots(figsize=(4.6, 3.8), dpi=300)
sns.violinplot(
    data=plot_df_all,
    x='Compartment',
    y='Y_score',
    order=comp_order_D,
    hue='Compartment',
    palette=comp_palette,
    inner='box',
    cut=0,
    linewidth=1,
    ax=ax,
)
leg = ax.get_legend()
if leg is not None:
    leg.remove()
ax.set_title('Y-score')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=90)
my_savefig(DIR_FIG_SAVE / 'FigD_Yscore_violin_by_compartment.pdf')
plt.close(fig)

# (E) Stacked bar: % LOY vs ROY in each compartment
counts_ce = (
    plot_df_all.groupby(['Compartment', 'Y_group'], observed=False)
    .size()
    .unstack(fill_value=0)
)
for col in ['LOY', 'ROY']:
    if col not in counts_ce.columns:
        counts_ce[col] = 0
counts_ce = counts_ce[['LOY', 'ROY']]
pct_ce = counts_ce.div(counts_ce.sum(axis=1).replace(0, np.nan), axis=0) * 100
pct_ce = pct_ce.loc[comp_order_E]

# Plot-ready exports for Fig E
counts_ce.loc[comp_order_E].to_csv(DIR_PLOT_DATA / 'FigE_counts_LOY_ROY_by_compartment.csv')
pct_ce.to_csv(DIR_PLOT_DATA / 'FigE_percent_LOY_ROY_by_compartment.csv')
pct_ce.to_csv(DIR_SAVE / 'FigE_loy_roy_percentage_by_compartment.tsv', sep='\t')

fig, ax = plt.subplots(figsize=(4.6, 3.8), dpi=300)
x = np.arange(len(pct_ce.index))
ax.bar(x, pct_ce['ROY'].fillna(0), color=ROY_COLOR, label='ROY')
ax.bar(x, pct_ce['LOY'].fillna(0), bottom=pct_ce['ROY'].fillna(0), color=LOY_COLOR, label='LOY')
ax.set_ylim(0, 100)
ax.set_ylabel('Percentage (%)')
ax.set_xticks(x)
ax.set_xticklabels(pct_ce.index, rotation=90)
ax.set_xlabel('')
ax.legend(title='', frameon=False, loc='upper right')
my_savefig(DIR_FIG_SAVE / 'FigE_LOYROY_stackedbar_by_compartment.pdf')
plt.close(fig)

# (F) Summary violin: metastatic capacity (scMetRate) in lineage-enriched LOY vs ROY populations
fig, ax = plt.subplots(figsize=(3.0, 3.8), dpi=300)
sns.violinplot(
    data=figf_df,
    x='LineageClass',
    y='scMetRate',
    order=['LOY', 'ROY'],
    hue='LineageClass',
    palette=LOYROY_PALETTE,
    inner='box',
    cut=0,
    linewidth=1,
    ax=ax,
)
leg = ax.get_legend()
if leg is not None:
    leg.remove()
ax.set_xlabel('LOY status (lineage-enriched)')
ax.set_ylabel('metastatic capacity (scMetRate)')
my_savefig(DIR_FIG_SAVE / 'FigF_scMetRate_violin_LOY_vs_ROY.pdf')
plt.close(fig)

# Optional: Combined panel figure (D/E/F)
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.8), dpi=300, gridspec_kw={'width_ratios': [1.35, 1.15, 0.8]})
axD, axE, axF = axes

sns.violinplot(
    data=plot_df_all, x='Compartment', y='Y_score', order=comp_order_D,
    hue='Compartment', palette=comp_palette,
    inner='box', cut=0, linewidth=1, ax=axD
)
leg = axD.get_legend()
if leg is not None:
    leg.remove()
axD.set_title('Y-score')
axD.set_xlabel('')
axD.tick_params(axis='x', rotation=90)

x = np.arange(len(pct_ce.index))
axE.bar(x, pct_ce['ROY'].fillna(0), color=ROY_COLOR, label='ROY')
axE.bar(x, pct_ce['LOY'].fillna(0), bottom=pct_ce['ROY'].fillna(0), color=LOY_COLOR, label='LOY')
axE.set_ylim(0, 100)
axE.set_ylabel('Percentage (%)')
axE.set_xticks(x)
axE.set_xticklabels(pct_ce.index, rotation=90)
axE.set_xlabel('')
axE.legend(title='', frameon=False, loc='upper right')

sns.violinplot(
    data=figf_df, x='LineageClass', y='scMetRate', order=['LOY', 'ROY'],
    hue='LineageClass', palette=LOYROY_PALETTE,
    inner='box', cut=0, linewidth=1, ax=axF
)
leg = axF.get_legend()
if leg is not None:
    leg.remove()
axF.set_xlabel('LOY status (lineage-enriched)')
axF.set_ylabel('metastatic capacity (scMetRate)')

for label, ax in zip(['D', 'E', 'F'], [axD, axE, axF]):
    ax.text(-0.18, 1.08, label, transform=ax.transAxes, fontsize=16, fontweight='bold', va='top')

fig.tight_layout(w_pad=2.0)
my_savefig(DIR_FIG_SAVE / 'Fig6DEF_main.pdf')
plt.close(fig)


## Supplementary figures
Additional visualizations (UMAP/score plots and correlation plots) are generated in `03_A549_FigS6_Supplementary.ipynb`.

## 9. LOY/ROY statistics
Chi-square across groups and Fisher tests vs reference.

In [ ]:
chi2, p_value, dof, expected = chi_square_loy_roy(counts)
print(f'Chi-square p-value: {p_value:.3e}')

pairwise = fisher_pairwise_vs_reference(counts, reference=REFERENCE_GROUP, low_label='LOY', high_label='ROY')
pairwise.to_csv(DIR_SAVE / f'fisher_vs_{REFERENCE_GROUP}.csv')
print(pairwise.sort_values('p_value').head())
